# UERC26 — Raw DINOv2 Embeddings (No Training)

Extracts **raw pretrained DINOv2** embeddings and evaluates vs baseline.
No training. Just: image → DINOv2 → embedding → cosine similarity.

> **Runtime:** Runtime → Change runtime type → **T4 GPU**

## Cell 1: Mount Google Drive & Setup

In [ ]:
from google.colab import drive
import zipfile, os

drive.mount('/content/drive')

# === EDIT THIS PATH to match where your zip is in Drive ===
ZIP_PATH = '/content/drive/MyDrive/UERC26_toolkit_baseline.zip'
# If your zip has a different name, change it above.
# For example: '/content/drive/MyDrive/Colab/uerc26.zip'

if not os.path.exists(ZIP_PATH):
    print(f"ERROR: Zip not found at {ZIP_PATH}")
    print("Available files in MyDrive root:")
    for f in os.listdir('/content/drive/MyDrive/')[:20]:
        if 'uerc' in f.lower() or 'UERC' in f:
            print(f"  >>> {f}  <-- maybe this one?")
        else:
            print(f"  {f}")
else:
    print(f"Unzipping {ZIP_PATH}...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')
    
    # Find extracted folder
    for folder in os.listdir('/content/'):
        if 'UERC26' in folder or 'uerc' in folder.lower():
            if os.path.isdir(f'/content/{folder}'):
                PROJECT_DIR = f'/content/{folder}'
                break
    
    os.chdir(PROJECT_DIR)
    print(f"Project directory: {PROJECT_DIR}")
    print(f"Files: {[f for f in os.listdir('.') if not f.startswith('.')]}")

## Cell 2: Install Dependencies & Check GPU

In [ ]:
%pip install opencv-python-headless scipy -q

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU! Runtime -> Change runtime type -> T4 GPU")

## Cell 3: Extract Raw Embeddings (~5-10 min on GPU)

In [ ]:
import os, sys, csv, json, time
import numpy as np
from PIL import Image
import cv2
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.notebook import tqdm

os.chdir(PROJECT_DIR)
from ear_preprocessing import normalize_ear_side, apply_clahe

class SimpleEarDataset(Dataset):
    def __init__(self, root_dir, split_csv='dataset_split.csv', split=None):
        self.root_dir = root_dir
        self.images = []
        split_path = os.path.join(root_dir, split_csv)
        if os.path.exists(split_path):
            with open(split_path, 'r') as f:
                for row in csv.reader(f, delimiter='\t'):
                    if len(row) >= 2 and (split is None or row[1] == split):
                        self.images.append(row[0])
        else:
            for dp, _, fns in os.walk(root_dir):
                for fn in sorted(fns):
                    if fn.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.images.append(os.path.relpath(os.path.join(dp, fn), root_dir).replace('\\', '/'))
        self.transform = transforms.Compose([
            transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        # Load raw image without preprocessing as requested previously
        img = cv2.imread(os.path.join(self.root_dir, self.images[idx]))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = normalize_ear_side(img, target_side='right')
        img = apply_clahe(img)
        return self.transform(Image.fromarray(img)), self.images[idx]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# CHANGED: Switched from dinov2_vits14 to dinov2_vitl14
print("Loading DINOv2 ViT-L/14 (Large)...")
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14', pretrained=True)
model = model.to(device).eval()
print("Model loaded!\n")

PUBLIC_ROOT = 'UERC26_data/data/public'
SEQ_ROOT = 'UERC26_data/data/sequestered_anonymized'
OUTPUT_DIR = 'raw_embeddings_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

all_features = {}
for split in ['val', 'test']:
    ds = SimpleEarDataset(PUBLIC_ROOT, split=split)
    print(f"[{split.upper()}] {len(ds)} images")
    # Batch size reduced to 32 because DINOv2 Large is significantly heavier on VRAM
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
    t0 = time.time()
    with torch.no_grad():
        for imgs, names in tqdm(loader, desc=split):
            embs = F.normalize(model(imgs.to(device)), p=2, dim=1)
            for name, emb in zip(names, embs.cpu().numpy()):
                all_features[name] = emb
    print(f"  Done in {time.time()-t0:.0f}s")

if os.path.exists(SEQ_ROOT):
    seq_ds = SimpleEarDataset(SEQ_ROOT)
    print(f"\n[SEQUESTERED] {len(seq_ds)} images")
    seq_loader = DataLoader(seq_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
    seq_features = {}
    with torch.no_grad():
        for imgs, names in tqdm(seq_loader, desc='sequestered'):
            embs = F.normalize(model(imgs.to(device)), p=2, dim=1)
            for name, emb in zip(names, embs.cpu().numpy()):
                seq_features[name] = emb
    np.save(os.path.join(OUTPUT_DIR, 'sequestered_features.npy'), seq_features)

print(f"\nTotal: {len(all_features)} public + {len(seq_features)} sequestered embeddings")

## Cell 4: Evaluate & Compare with Baseline

In [ ]:
import matplotlib.pyplot as plt
from uerc26_utils import compute_metrics

BASELINE = {'VER@0.1%': 0.2623, 'VER@1%': 0.4631, 'EER': 0.1569, 'AUC': 0.9215}
results = {}

for split_name, pairs_csv in [('VAL', 'pairs_val.csv'), ('TEST', 'pairs_test.csv')]:
    pairs_path = os.path.join(PUBLIC_ROOT, pairs_csv)
    if not os.path.exists(pairs_path): continue
    scores, labels = [], []
    with open(pairs_path, 'r') as f:
        for row in csv.reader(f, delimiter='\t'):
            if len(row) >= 3 and row[0] in all_features and row[1] in all_features:
                f1, f2 = all_features[row[0]], all_features[row[1]]
                scores.append(np.dot(f1, f2) / (np.linalg.norm(f1) * np.linalg.norm(f2) + 1e-8))
                labels.append(int(row[2]))
    metrics = compute_metrics(np.array(labels), np.array(scores))
    results[split_name] = metrics
    print(f"\n{'='*55}")
    print(f"  {split_name} — Raw DINOv2 (NO training) vs Baseline")
    print(f"{'='*55}")
    for k in ['VER@0.1%', 'VER@1%', 'EER', 'AUC']:
        b, y = BASELINE[k]*100, metrics[k]*100
        d = y - b
        s = 'BETTER' if (d > 0 and k != 'EER') or (d < 0 and k == 'EER') else 'WORSE'
        print(f"  {k:<12} Baseline: {b:6.2f}%  Yours: {y:6.2f}%  ({d:+.2f}%) {s}")
    with open(os.path.join(OUTPUT_DIR, f'{split_name.lower()}_metrics.json'), 'w') as f:
        json.dump(metrics, f, indent=4)

if 'TEST' in results:
    m = results['TEST']
    names = ['VER@0.1%', 'VER@1%', 'AUC']
    fig, ax = plt.subplots(figsize=(10, 5))
    x = range(len(names)); w = 0.35
    ax.bar([i-w/2 for i in x], [BASELINE[k]*100 for k in names], w, label='Baseline (trained)', color='#e74c3c', alpha=0.85)
    ax.bar([i+w/2 for i in x], [m[k]*100 for k in names], w, label='Raw DINOv2 (no training)', color='#2ecc71', alpha=0.85)
    ax.set_ylabel('Score (%)'); ax.set_title('Raw DINOv2 vs Trained Baseline', fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(names); ax.legend(); ax.set_ylim(0, 100)
    plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, 'comparison.png'), dpi=150); plt.show()

## Cell 5: Save Results to Drive

In [ ]:
import shutil
drive_dest = '/content/drive/MyDrive/uerc26_raw_results'
if os.path.exists(drive_dest):
    shutil.rmtree(drive_dest)
shutil.copytree(OUTPUT_DIR, drive_dest)
print(f"Results saved to Google Drive: {drive_dest}")
for f in os.listdir(drive_dest):
    print(f"  {f} ({os.path.getsize(os.path.join(drive_dest, f))/1e6:.1f} MB)")